# 2. Text Preprocessing and Feature Extraction
## AG News Dataset Preprocessing

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from src.preprocessing import TextPreprocessor, FeatureExtractor
from src.utils import save_model

import warnings
warnings.filterwarnings('ignore')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...


## Load Data

In [2]:
train_df = pd.read_csv('../data/processed/train_explored.csv')
test_df = pd.read_csv('../data/processed/test_explored.csv')

print(f"Train: {len(train_df)}, Test: {len(test_df)}")

Train: 120000, Test: 7600


In [ ]:
# run this in case text cleaning cause LoopupError
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

## Text Cleaning

In [5]:
# Initialize preprocessor
preprocessor = TextPreprocessor(remove_stopwords=True, lemmatize=True)

# Example preprocessing
sample_text = train_df['text'].iloc[0]
print("Original:")
print(sample_text)
print("\nCleaned:")
print(preprocessor.preprocess(sample_text))  # example of cleaning

Original:
Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.

Cleaned:
wall st bear claw back black reuters reuters shortsellers wall street dwindlingband ultracynics seeing green


In [6]:
# Apply to all data
print("Preprocessing training data...")
train_df = preprocessor.preprocess_dataframe(train_df)

print("Preprocessing test data...")
test_df = preprocessor.preprocess_dataframe(test_df)

print("Done!")

Preprocessing training data...
Preprocessing test data...
Done!


## Feature Extraction - TF-IDF

In [7]:
# Extract TF-IDF features
tfidf_extractor = FeatureExtractor(method='tfidf', max_features=5000)

X_train_tfidf = tfidf_extractor.fit_transform(train_df['cleaned_text'])
X_test_tfidf = tfidf_extractor.transform(test_df['cleaned_text'])

print(f"TF-IDF features shape - Train: {X_train_tfidf.shape}, Test: {X_test_tfidf.shape}")

TF-IDF features shape - Train: (120000, 5000), Test: (7600, 5000)


## Feature Extraction - Count Vectorizer

In [8]:
# Extract count features
count_extractor = FeatureExtractor(method='count', max_features=5000)

X_train_count = count_extractor.fit_transform(train_df['cleaned_text'])
X_test_count = count_extractor.transform(test_df['cleaned_text'])

print(f"Count features shape - Train: {X_train_count.shape}, Test: {X_test_count.shape}")

Count features shape - Train: (120000, 5000), Test: (7600, 5000)


## Save Preprocessed Data and Vectorizers

In [9]:
import joblib

# Save data
train_df.to_csv('../data/processed/train_preprocessed.csv', index=False)
test_df.to_csv('../data/processed/test_preprocessed.csv', index=False)

# Save vectorizers
save_model(tfidf_extractor, '../models/tfidf_vectorizer.pkl')
save_model(count_extractor, '../models/count_vectorizer.pkl')

# Save features
joblib.dump(X_train_tfidf, '../data/processed/X_train_tfidf.pkl')
joblib.dump(X_test_tfidf, '../data/processed/X_test_tfidf.pkl')
joblib.dump(X_train_count, '../data/processed/X_train_count.pkl')
joblib.dump(X_test_count, '../data/processed/X_test_count.pkl')

# Save labels
y_train = train_df['class'].values
y_test = test_df['class'].values
joblib.dump(y_train, '../data/processed/y_train.pkl')
joblib.dump(y_test, '../data/processed/y_test.pkl')

print("All preprocessed data saved!")

Model saved to ../models/tfidf_vectorizer.pkl
Model saved to ../models/count_vectorizer.pkl
All preprocessed data saved!
